## Practising everything learnt until now

### 1- Initialization of the model necessary libraries and logging

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List
from pydantic import BaseModel, Field
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)


### Level-1: Foundations (Graph Basics)

In [3]:
class state_schema(TypedDict):
    input:str

graph = StateGraph(state_schema)

def hello_world(state: state_schema) -> state_schema:
    llm = model.invoke(state["input"])
    print(llm.content)
    return {"input":llm.content}
#adding nodes
graph.add_node("hello_world", hello_world)

#adding edges
graph.add_edge(START, "hello_world")
graph.add_edge("hello_world", END)

app = graph.compile()

if __name__ == "__main__":
    result = app.invoke({"input":"python"})
    print(result)


2026-04-20 19:26:07,627 - INFO - AFC is enabled with max remote calls: 10.


2026-04-20 19:26:09,620 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"
2026-04-20 19:26:09,626 - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
2026-04-20 19:26:13,065 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"
2026-04-20 19:26:13,071 - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 2.05 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### 2-Creating a Multi-Step flow

In [26]:
class state_schema(TypedDict):
    input: str
    response: str

def create_flow(state: state_schema) -> dict:
    llm_response = model.invoke(state["input"])
    state["response"] = llm_response.content
    return {"response": state["response"]   }

graph = StateGraph(state_schema)
graph.add_node("create_flow", create_flow)
graph.add_edge(START, "create_flow")
graph.add_edge("create_flow", END)

app = graph.compile()

if __name__ == "__main__":
    user_input = input("Enter your query: ")

    messages = [
        ("system", "You are a helpful assistant that provides concise answers to user queries."),
        ("human", user_input)
    ]

    result = app.invoke({"input": messages})
    print(result["response"])

2026-04-19 22:09:14,369 - INFO - AFC is enabled with max remote calls: 10.
2026-04-19 22:09:16,475 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. It encompasses learning, problem-solving, perception, and decision-making.


### Conditional Routing(if-else agent)

In [7]:
class state_schema(TypedDict):
    input: str
    response: str
    intent: str

def classify(state: state_schema) -> dict:
    prompt = f"""Classify the intent of the following user query into one of the following categories: 'task', 'question'.
    input: {state["input"]}"""
    llm_response = model.invoke(prompt)
    intent = llm_response.content.strip().lower()
    if "question" in intent:
        intent = "question"
    else:
        intent = "task"

    return {"intent": intent}

def explain(state: state_schema) -> dict:
    prompt = f"""Explain the following user query in detail: {state["input"]}"""
    llm_response = model.invoke(prompt)
    return {"response": llm_response.content}

def task(state: state_schema) -> dict:
    prompt = f"""Generate content on : {state["input"]}"""
    llm_response = model.invoke(prompt)
    return {"response": llm_response.content}

def route(state: state_schema) -> str:
    if state["intent"] == "question":
        return "explain"
    else:
        return "task"

graph = StateGraph(state_schema)
graph.add_node("classify", classify)
graph.add_node("explain", explain)
graph.add_node("task", task)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {"explain": "explain", "task": "task"})
graph.add_edge("explain", END)
graph.add_edge("task", END)

app = graph.compile()

if __name__ == "__main__":
    user_input = input("Enter your query: ")

    result = app.invoke({"input": user_input})
    print(result["response"])



2026-04-20 20:55:53,842 - INFO - AFC is enabled with max remote calls: 10.
2026-04-20 20:57:20,771 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-20 20:57:20,788 - INFO - AFC is enabled with max remote calls: 10.
2026-04-20 20:57:42,093 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


The user query "is python a markdown language" is a straightforward question asking for a classification of Python in relation to Markdown.

Here's a detailed explanation of the query and its answer:

---

## Detailed Explanation of the User Query: "is python a markdown language"

The user is asking if Python, a programming language, belongs to the category of "markdown languages." This query stems from a fundamental misunderstanding or a conflation of concepts, likely due to how Python and Markdown are often used *together* in various contexts (like documentation, web development, or data science notebooks).

Let's break down each term and then address the relationship:

### 1. What is Python?

*   **Type:** Python is a **high-level, general-purpose, interpreted programming language.**
*   **Purpose:** Its primary purpose is to write **executable code** – instructions that a computer can understand and follow to perform tasks, solve problems, build applications, analyze data, automate

### calculator Agent

In [ ]:
class state_schema(TypedDict):
    input: str
    is_math: bool
    output: str

def decide(state: state_schema)-> dict:
    query = state["input"].lower()
    if any(keyword in query for keyword in ["calculate", "solve", "evaluate"]):
        state["is_math"] = True
    return {"is_math": state["is_math"]}